<a href="https://colab.research.google.com/github/somendrew/LangGraph_tutorial/blob/main/7_HITL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![HITL Concept](images/hitl_concept.svg)





*   Step 1 — A normal graph (no human-in-the-loop yet)
*   Step 2 — Add a checkpointer (this is what makes pausing possible)
*   Step 3 — Tell it WHERE to pause with interrupt_before
*   Step 4 — Resume the graph (human gives the go-ahead)





![HITL Full Flow](images/hitl_full_flow.svg)


# LangGraph Checkpointing & Interruptions Cheat Sheet
| Component / Parameter | What it does | Where it appears in code |
| :--- | :--- | :--- |
| **`MemorySaver()`** | Create the save slot. | `memory = MemorySaver()` |
| **`checkpointer`** | Attaches the save-slot graph | `compile(checkpointer=memory)` |
| **`interrupt_before`** | Says where to pause | `compile(interrupt_before=["send"])` |
| **`thread_id`** | The name of your save-file | `config = {"configurable": {"thread_id": "abc"}}` |


And then:


* First invoke() → runs up to the pause, stops
* get_state() → read what's in there
* update_state() → change it if you want
* Second invoke(None) → same thread_id → continues from where it stopped


In [1]:
!pip install -q langgraph langchain_openai langchain_core typing

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 6.6 MB/s eta 0:00:00


In [6]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

class EmailState(TypedDict):
    draft: str

def draft_email(state: EmailState) -> EmailState:
    print("AI: Writing Draft")
    return {"draft": "Hi! Just checking in."}

def send_email(state: EmailState):
    print(f"\nSENT: {state['draft']}")
    return state

builder = StateGraph(EmailState)
builder.add_node(draft_email)
builder.add_node(send_email)
builder.add_edge(START, "draft_email")
builder.add_edge("draft_email", "send_email")
builder.add_edge("send_email", END)

memory = MemorySaver()
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["send_email"]
)

config = {"configurable": {"thread_id": "email-001"}}

print("=== Starting graph ===")
graph.invoke({"draft": ""}, config=config)          # ✅ state key

current = graph.get_state(config)
print(f"\n=== Graph paused. Draft: ===\n{current.values['draft']}")  # ✅ state key

human_edit = input("\nEdit the draft (or press Enter to approve as-is): ")
if human_edit.strip():
    graph.update_state(config, {"draft": human_edit})  # ✅ state key
    print("Draft updated.")

print("\n=== Resuming graph ===")
graph.invoke(None, config=config)

=== Starting graph ===
AI: Writing Draft

=== Graph paused. Draft: ===
Hi! Just checking in.

Edit the draft (or press Enter to approve as-is): This is new email # Updated
Draft updated.

=== Resuming graph ===

SENT: This is new email # Updated


{'draft': 'This is new email # Updated'}